# Módulo 0: Pré-requisitos & Configuração do Ambiente

---

## Bem-vindo ao Laboratório Prático do Amazon Bedrock AgentCore!

Neste curso prático, você vai construir a **Aria** -- uma assistente de inteligência artificial pronta para produção usando o **Amazon Bedrock AgentCore**. Ao longo de 9 módulos, você vai do zero até ter uma IA totalmente no ar, segura e com métricas de observabilidade.

![Aria AI Chat](../shared/img/aria-home.png)

### O que você vai construir

A Aria não é apenas um projetinho de teste. Até o final deste curso, sua assistente vai:

- **Rodar no AgentCore Runtime** -- hospedada como um endpoint gerenciado e escalável
- **Usar o Interpretador de Código e o Navegador Web** -- para rodar código Python e navegar na internet por conta própria
- **Lembrar das conversas** -- usando o AgentCore Memory para manter o contexto mesmo depois de fechar a sessão
- **Ter uma API segura** -- acessível pelo AgentCore Gateway com autenticação baseada em identidade (JWT)
- **Respeitar regras de negócio** -- aplicando bloqueios rígidos usando políticas Cedar
- **Gerar traces e métricas** -- acompanhando tudo pelo AgentCore Observabilidade
- **Passar por avaliações automáticas** -- sendo testada por outras IAs usando o AgentCore Avaliações

![Aria AI Demo](../shared/img/aria-demo.png)

### Módulos do Curso

| Módulo | Tópico |
|--------|-------|
| **00** | Pré-requisitos & Configuração do Ambiente (este módulo) |
| **01** | Introdução ao Amazon Bedrock AgentCore |
| **02** | Fazendo o Deploy do seu Primeiro Agente |
| **03** | Adicionando Ferramentas: Código e Navegador |
| **04** | Dando Memória Persistente para o Agente |
| **05** | Conectando o Gateway & Autenticação |
| **06** | Bloqueios e Regras com Políticas Cedar |
| **07** | Observabilidade e Avaliações de Qualidade |
| **08** | Subindo Tudo em Produção (Frontend) |

Vamos começar checando se o seu ambiente está pronto.

---

## 1. Verificando o Ambiente

Precisamos confirmar se as ferramentas estão instaladas e se o seu acesso na AWS está configurado certinho.

### 1.1 Versão do Python

Este curso exige o **Python 3.12**.

In [ ]:
!python3 --version

### 1.2 AWS CLI

A linha de comando da AWS (CLI) é usada para vários comandos durante o laboratório.

In [ ]:
!aws --version

### 1.3 AWS CDK

O AWS CDK vai ser usado nos módulos de deploy.

In [ ]:
!cdk --version

### 1.4 Credenciais da AWS

Verifique se suas credenciais da AWS estão válidas e se consegue bater na API.

In [ ]:
import sys
# Adiciona o diretório pai ao PATH do Python para permitir imports relativos.
sys.path.insert(0, '..')

# Importa o SDK da AWS para Python — usado para interagir com todos os serviços da AWS.
import boto3

sts = boto3.client('sts')
# Verifica qual conta e usuário da AWS estão configurados no momento.
identity = sts.get_caller_identity()

print(f"Account:  {identity['Account']}")
print(f"Arn:      {identity['Arn']}")
print(f"UserId:   {identity['UserId']}")
print("\nAWS credentials are valid.")

---

## 1.5 X-Ray Transaction Search Check

The prerequisites CloudFormation stack enables **X-Ray Transaction Search** by default. This is an account-level setting (only one configuration is allowed per account per region), and it is required for the observability module later in the workshop.

If your account already has Transaction Search enabled, the stack deployment will fail unless you set the `EnableTransactionSearch` parameter to `false`. Run the cell below to check your account's current status **before deploying the stack**.

In [ ]:
import boto3
from botocore.exceptions import ClientError

region = "us-east-1"
xray = boto3.client("xray", region_name=region)

try:
    response = xray.get_indexing_rules()
    indexing_rules = response.get("IndexingRules", [])

    enabled = False
    for rule in indexing_rules:
        probabilistic = rule.get("Rule", {}).get("Probabilistic", {})
        if probabilistic.get("DesiredSamplingPercentage", 0) > 0:
            enabled = True
            pct = probabilistic["DesiredSamplingPercentage"]
            break

    if enabled:
        print(f"✅ X-Ray Transaction Search is ENABLED (indexing {pct}% of traces).")
        print()
        print("   Your account already has Transaction Search configured.")
        print("   When deploying the prerequisites stack, add this parameter")
        print("   to skip creating a duplicate configuration:")
        print()
        print("   aws cloudformation deploy \\")
        print("     --template-file infrastructure/prerequisites.yaml \\")
        print("     --stack-name agentcore-workshop-prerequisites \\")
        print("     --capabilities CAPABILITY_NAMED_IAM \\")
        print("     --parameter-overrides EnableTransactionSearch=false \\")
        print("     --region us-east-1")
    else:
        print("ℹ️  X-Ray Transaction Search is NOT currently enabled in this account/region.")
        print()
        print("   The prerequisites stack will enable it automatically (default behavior).")
        print("   No extra parameters needed -- just deploy with the standard command:")
        print()
        print("   aws cloudformation deploy \\")
        print("     --template-file infrastructure/prerequisites.yaml \\")
        print("     --stack-name agentcore-workshop-prerequisites \\")
        print("     --capabilities CAPABILITY_NAMED_IAM \\")
        print("     --region us-east-1")

except ClientError as e:
    print(f"⚠️  Could not check Transaction Search status: {e}")
    print()
    print("   If the stack deployment fails on the TransactionSearchConfig resource,")
    print("   redeploy with --parameter-overrides EnableTransactionSearch=false")
except Exception as e:
    print(f"⚠️  Unexpected error checking Transaction Search: {e}")
    print()
    print("   If the stack deployment fails on the TransactionSearchConfig resource,")
    print("   redeploy with --parameter-overrides EnableTransactionSearch=false")

---

## 2. CloudFormation Stack Verification

If you followed the README instructions, you deployed the prerequisites CloudFormation stack using `aws cloudformation deploy`. Let's verify that the stack completed successfully and that all required outputs are available.

In [ ]:
import sys
# Adiciona o diretório pai ao PATH do Python para permitir imports relativos.
sys.path.insert(0, '..')

# Verifica se a stack do CloudFormation foi criada e lista os recursos disponíveis.
from shared.progress import check_prerequisites

# Verifica se a stack do CloudFormation foi criada e lista os recursos disponíveis.
check_prerequisites()

---

## 3. What Was Provisioned

The CloudFormation stack created the following resources that Aria will use throughout the workshop:

### Authentication & Identity

- **Amazon Cognito User Pool** -- Manages user accounts for Aria's end users
- **Cognito App Client** -- Allows Aria's frontend to authenticate users
- **Cognito Domain** -- Provides hosted UI endpoints for sign-in flows

### Data & APIs

- **Amazon DynamoDB Table** -- Stores tasks that Aria can create, read, update, and delete
- **AWS Lambda Function** -- Implements the Task API business logic
- **Amazon API Gateway REST API** -- Exposes the Task API as a secure HTTP endpoint

### Storage

- **Amazon S3 Bucket** -- Stores artifacts, logs, and other files generated during the workshop

### IAM Roles

- **Runtime Execution Role** -- Grants Aria's agent the permissions it needs when running in AgentCore Runtime
- **Gateway Execution Role** -- Grants AgentCore Gateway the permissions to invoke and manage agent endpoints

### Observability

- **X-Ray Transaction Search** -- Enables distributed trace indexing so you can search and analyze agent traces in the observability module. This is an account-level setting that indexes 100% of traces and takes approximately 10 minutes to become active. If your account already had Transaction Search enabled, this resource was skipped (controlled by the `EnableTransactionSearch` stack parameter).
- **CloudWatch Logs Resource Policy** -- Grants X-Ray permission to write trace spans to CloudWatch Logs (`aws/spans` log group)

These resources form the backbone of Aria's environment. In the modules ahead, you will connect Aria to each of them.

---

## 4. Marcar o Módulo como Concluído

Tudo certo! Vamos salvar o seu progresso.

In [ ]:
import sys
# Adiciona o diretório pai ao PATH do Python para permitir imports relativos.
sys.path.insert(0, '..')

# Importa as ferramentas compartilhadas do workshop (funções auxiliares).
from shared.progress import show

show("00")

---

**Próximo Passo: [Módulo 1 -- Introdução ao Amazon Bedrock AgentCore](../01-introduction/notebook.ipynb)**